In [ ]:
from typing import Annotated

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START , END

from langgraph.graph.message import add_messages


In [ ]:
class state(TypedDict):
    messages:Annotated[list,add_messages]

In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

llm=ChatGroq(model="llama3-8b-8192")

In [ ]:
llm

In [ ]:
llm = init_chat_model("groq:llama3-8b-8192")
llm

In [ ]:
def chatbot(state:state):
    return {"messages":[llm.invoke(state["messages"])]}

In [ ]:
graph_builder = StateGraph(state)

graph_builder.add_node("llmchatbot",chatbot)

graph_builder.add_edge(START,"llmchatbot")
graph_builder.add_edge("llmchatbot",END)


graph=graph_builder.compile()

In [ ]:
response = graph.invoke({"messages":"Hi"})


In [ ]:
response["messages"][-1].content

In [ ]:
from langchain_tavily import TavilySearch

tool = TavilySearch(max_result=2)
tool.incoke("what is langgraph")

In [ ]:
def multiply(a:int,b:int)->int:
   """
   Multiple a and b

   Args:
       a(int):first int
       b(int):second int

    Returns:
        int; output int
    
    """
   return a*b

In [ ]:
tools=[tool,multiply]

In [ ]:
llm_with_tool=llm.bind_tools(tools)

In [ ]:
llm_with_tool

In [ ]:
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition


def tool_calling_llm(state:state):
    return {"messages":[llm_with_tool.invoke(state["message"])]}

builder=StateGraph(state)
builder.add_node("tool_calling_llm",tool_calling_llm)
builder.add_node("tool",ToolNode(tool))

builder.add_edge(START,"tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm"
     tools_condition
)

builder.add_edge('tools',END)

graph=builder.compile()



In [ ]:
response=graph.invoke({"messages":"What is the recent ai news"})